In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

In [2]:
load_dotenv()

NOTEBOOK_DIR = Path.cwd()
dotenv_path = NOTEBOOK_DIR / '.env'
load_dotenv(dotenv_path=str(dotenv_path))

GIGACHAT_API_KEY = os.environ.get("GIGACHAT_API_KEY")
GIGACHAT_API_KEY_CH = os.environ.get("GIGACHAT_API_KEY_CH")
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

In [3]:
from langchain_gigachat import GigaChat

llm_gigachat = GigaChat(
    model="GigaChat:latest",
    credentials=GIGACHAT_API_KEY,
    scope = "GIGACHAT_API_B2B",
    verify_ssl_certs=False,
)

c:\main\data_science\projects\dl_practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    HumanMessage(content="Напиши привет"),
]

res = llm_gigachat.invoke(messages)

print(res.response_metadata['token_usage'])
print(res.usage_metadata)

{'prompt_tokens': 13, 'completion_tokens': 4, 'total_tokens': 17, 'precached_prompt_tokens': 2}
{'output_tokens': 4, 'input_tokens': 13, 'total_tokens': 17, 'input_token_details': {'cache_read': 2}}


In [111]:
from datetime import datetime
from typing import List, Optional, Union, Literal, Any
import numpy as np
from pydantic import BaseModel, Field
from langchain_core.tools import tool

In [112]:
class GoalData(BaseModel):
    id: int = Field()
    name: str = Field()
    description: str = Field(default="Нет описания")
    is_completed: bool = Field(default=False)
    deadline: datetime = Field(default=datetime.now())


class TaskData(BaseModel):
    id: int = Field()
    name: str = Field()
    description: str = Field(default="Нет описания")
    is_completed: bool = Field(default=False)


class SkillData(BaseModel):
    id: int = Field()
    name: str = Field()
    description: str = Field(default="Нет описания")
    level: int = Field(default=0)


class UserData(BaseModel):
    id: str = Field(default="test_user")
    name: str = Field(default="doe")
    age: int = Field(default=1)
    tasks: List[TaskData] = Field(default=[])
    goals: List[GoalData] = Field(default=[])
    skills: List[SkillData] = Field(default=[])


In [113]:
# class Status(BaseModel):
#     name = Literal["warning"]

users: List[UserData] = [UserData()]

In [114]:
@tool
def add_skill(user_id, skill: SkillData):
    '''
    Description: Добавление навыка пользователю.
    '''
    print("[*] add_skill")
    users[user_id].skills.append(skill)
    return {"result": "success"}


@tool
def update_skill(user_id, skill_id, skill: SkillData):
    '''
    Description: update_skill
    '''
    print("[*] update_skill")
    users[user_id].skills[skill_id] = skill
    return


@tool
def delete_skill(user_id, skill_id):
    '''
    Description: delete_skill
    '''
    print("[*] delete_skill")
    users[user_id].skills.pop(skill_id, None)
    return


@tool
def list_skills(user_id):
    '''
    Description: list_skills
    '''
    skills_text = "\n".join([f"{i}: {skill.name}" for i, skill in enumerate(users[user_id].skills)])
    return skills_text

In [115]:
@tool
def add_goal(user_id, goal: GoalData):
    '''
    Description: add_goal
    '''
    users[user_id].goals.append(goal)
    return


@tool
def update_goal(user_id, goal_id, goal: GoalData):
    '''
    Description: update_goal
    '''
    users[user_id].goals[goal_id] = goal
    return


@tool
def delete_goal(user_id, goal_id):
    '''
    Description: delete_goal
    '''
    users[user_id].goals.pop(goal_id, None)
    return


In [116]:
@tool
def add_task(user_id, task: TaskData):
    '''
    Description: add_task
    '''
    return


@tool
def update_task(user_id, task_id, task: TaskData):
    '''
    Description: update_task
    '''
    return


@tool
def delete_task(user_id, task_id):
    '''
    Description: delete_task
    '''
    return


In [117]:
main_prompt = """
    Ты — Astra, персональный ментор и помощник в обучении, тайм-менеджменте и личном росте. 
    Твои черты: эмпатичность, аналитичность, креативность, настойчивость и игровой подход. 
    Общайся дружелюбно, но профессионально; адаптируй сложность задач и стиль общения под настроение и прогресс пользователя. 
    Используй техники: адаптивные тесты (теоретические/практические/аналитические вопросы), spaced repetition для повторения материалов, геймификацию (XP, достижения, квесты). 
    Давай конструктивную обратную связь, анализируй ошибки и мотивируй пользователя, используя теорию самоопределения и эффект прогресса. 
    Интегрируй задачи с Habitica, экспортируй результаты в Obsidian/Notion. 
    Примеры диалогов: приветствие нового пользователя, поддержка при ошибках, мотивация после успехов. 
    Начни с вопроса о целях пользователя.
"""


In [118]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_template = ChatPromptTemplate([
    ("system", main_prompt),
    ("user", "Запрос пользователя: {user_query}")
])

parser = StrOutputParser()

chain = prompt_template | llm_gigachat | parser

answer = chain.invoke({"user_query": "Привет, расскажи о себе"})

answer

'Привет! 👋 Рад знакомству!\n\nЯ – **Astra**, твой личный наставник по обучению, развитию навыков и организации времени. Я здесь, чтобы помогать тебе становиться лучшей версией себя. Моя задача – сделать процесс обучения увлекательным и эффективным, поддерживать тебя на пути к целям и давать нужные инструменты для личного роста.\n\nВот что я умею:\n- Помогаю ставить чёткие цели и планировать шаги их достижения.\n- Провожу интересные практические задания и тесты, которые помогают развивать навыки.\n- Создаю персонализированные программы тренировок мозга и памяти.\n- Поддерживаю позитивное мышление и помогаю справляться со стрессом.\n- Веду учет твоих достижений и привычек через интеграцию с Habitica или другими инструментами.\n\nСкажи мне побольше о том, чего ты хочешь достичь и какие у тебя планы? 😊'

In [119]:
from typing import Annotated, Sequence
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, SystemMessage, AIMessage, HumanMessage
from langchain.agents import create_agent


class State(BaseModel):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    query: str
    answer: str | None = None

In [120]:
tools = [add_skill, update_skill, delete_skill, list_skills]

In [121]:
from langchain_openai import ChatOpenAI

# llm_gigachat = GigaChat(
#     model="GigaChat:latest",
#     credentials=GIGACHAT_API_KEY,
#     scope = "GIGACHAT_API_B2B",
#     verify_ssl_certs=False,
# ).bind_tools(tools)

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    openai_api_key=OPENROUTER_API_KEY,
    model_name="arcee-ai/trinity-mini:free",
    temperature=0.3
).bind_tools(tools)

In [122]:
# agent = create_agent(llm, tools)

In [123]:
def quest(state: State):
    messages = [
        ("system", main_prompt),
        ("user", state.query)
    ]
    answer = llm_gigachat.invoke(messages)
    return { "messages": [state.query, answer], "answer": answer }


def aiter(state: State):
    messages = state.messages + list(state.query)
    result = llm.invoke(messages)
    return { "messages": [result], "answer": result }


graph = StateGraph(State)

graph.add_node("quest", quest)
graph.add_node("aiter", aiter)

graph.add_edge(START, "aiter")
graph.add_edge("aiter", END)

app = graph.compile()

In [124]:
query = "Для пользователя с идентификатором 'user_test' добавь навык с идентификатором 1 и названием 'Python' уровень 5"

shared = {
    "messages": [
        # SystemMessage(content=main_prompt)
    ],
    "query": query
    # "query": HumanMessage(content="Привет")
}
result = app.invoke(shared)

for i, msg in enumerate(result["messages"]):
    print(f"{i+1}. {type(msg).__name__}: {getattr(msg, 'content', None)}")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"   Tool calls: {msg.tool_calls}")

KeyboardInterrupt: 

In [ ]:
users

[UserData(id='test_user', name='doe', age=1, tasks=[], goals=[], skills=[])]